# Architect's Challenge 1.C — Build Tina v0

**Before you start:** select **Cell > Run All** to initialize the harness. Then build.

**Your task:** build Tina as a ReAct agent over `cortex-policies`. When you are ready, run `python3 /home/elastic/submit.py` from the Terminal — it runs the held-out set and records your results.

The harness provides `ToolRegistry`, `react_loop`, `llm_client`, and `es_client`. No tools are pre-registered and no memory is wired. The approach is yours.

**To submit:** create `/home/elastic/tina_agent.py` with a `run_tina(question: str, model: str) -> dict` function that returns `{'answer': str, 'cited_policy_ids': list, 'retrieved_policy_ids': list, 'tool_calls': int, 'model': str}`. Then run `submit.py` from the Terminal.

In [ ]:
# ── Harness setup ─────────────────────────────────────────────────────────────
import sys, os, json, pathlib, time

sys.path.insert(0, '/opt/ara/lib')
from tina.client import llm_client, es_client, model_fast, model_strong
from tina.tools import ToolRegistry
from tina.loop import react_loop

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

# Fix LLM proxy URL if protocol is missing
proxy = os.environ.get('LLM_PROXY_URL', '')
if proxy and not proxy.startswith('http'):
    os.environ['LLM_PROXY_URL'] = f'https://{proxy}'

client = llm_client()
es     = es_client()
FAST   = model_fast()
STRONG = model_strong()
TRACES = pathlib.Path(os.environ.get('ARA_TRACE_DIR', '/home/elastic/.traces'))
TRACES.mkdir(parents=True, exist_ok=True)

print('Harness ready.')
print(f'  FAST:   {FAST}')
print(f'  STRONG: {STRONG}')
print(f'  cortex-policies docs: {es.count(index="cortex-policies")["count"]}')

---
## Build your agent

This notebook is a scratch pad. Use it however works for you.

When your agent is ready, write it as a `run_tina(question, model)` function in `/home/elastic/tina_agent.py` and run `submit.py` from the Terminal.

In [ ]:
# ── YOUR WORK ── Build Tina ──────────────────────────────────────────────────
# Suggested structure:
#
# 1. Register search_policies as a tool on a ToolRegistry.
# 2. Call react_loop() or write your own loop.
# 3. Collect answer, cited_policy_ids, retrieved_policy_ids, tool_calls count.
#
# Example search_policies implementation:
# tools = ToolRegistry()
# @tools.register(
#     name='search_policies',
#     description='Search Cortex Bank AML policies',
#     parameters={'type':'object','properties':{'query':{'type':'string'}},'required':['query']}
# )
# def search_policies(query: str) -> list:
#     resp = es.search(index='cortex-policies',
#         body={'query': {'semantic': {'field': 'body_semantic', 'query': query}}}, size=3)
#     return [{'policy_id': h['_source'].get('policy_id', h['_id']),
#              'body': h['_source']['body'][:600]} for h in resp['hits']['hits']]
#
# Then use react_loop() or build your own.
pass

In [ ]:
# ── Test your agent on one question before submitting ─────────────────────────
# Replace with your actual function call:
# result = run_tina("What is Cortex Bank's CTR threshold?", FAST)
# print(result)

---
## Submit

When your agent is ready:

1. Save your `run_tina(question, model) -> dict` function to `/home/elastic/tina_agent.py`.
2. Switch to the **Terminal** tab.
3. Run: `python3 /home/elastic/submit.py`
4. Select **Check** in the sidebar.